# General time-series container: behavioral and visual checks

Run all cells in order from this branch checkout. These tests use real PyVista datasets and synthetic data, without downloads. The prototype remains experimental. Use a kernel with this checkout’s dependencies (`pyvista-validation==0.5.4`). Static render cells use the notebook backend; rendering needs a working VTK backend. An assertion failure stops validation. Interactive exploration is optional.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if root.name == 'exploration':
    root = root.parent
assert (root / 'exploration').is_dir(), 'Run from this branch root or exploration directory.'
sys.path.insert(0, str(root))

import numpy as np
import pyvista as pv

pv.OFF_SCREEN = True
print('PyVista:', pv.__version__)


In [ ]:
def expect_error(error_type, action):
    """Assert an operation rejects invalid input."""
    try:
        action()
    except error_type:
        return
    raise AssertionError(f'Expected {error_type.__name__}')


## Changing topology and exact selection

In [ ]:
from exploration.time_series import TimeSeries

meshes = [pv.Plane(i_resolution=2, j_resolution=1), pv.Cube(), pv.Sphere(theta_resolution=12, phi_resolution=8)]
series = TimeSeries.from_datasets([0.0, 0.25, 1.5], meshes)
assert len(series) == 3
assert series.times == (0.0, 0.25, 1.5)
assert series[0].n_points != series[-1].n_points
assert series.at_time(0.25).n_cells == meshes[1].n_cells
expect_error(KeyError, lambda: series.at_time(0.5))
expect_error(ValueError, lambda: TimeSeries.from_datasets([0, 0], meshes[:2]))
expect_error(IndexError, lambda: series[3])
expect_error(TypeError, lambda: series[True])
print('Selection and validation passed')

## Independent ownership

In [ ]:
expected = series[0].points.copy()
meshes[0].points[:] += 10
frame = series[0]
frame.points[:] -= 5
np.testing.assert_array_equal(series[0].points, expected)
print('Input and returned-frame mutation are isolated')

## Lazy reader adapter
A minimal reader double supplies real datasets. This checks the adapter contract, rather than a specific file format.

In [ ]:
class SyntheticReader:
    """Supply synthetic real meshes through the temporal reader contract."""
    time_values = [0.0, 0.25, 1.5]

    def __init__(self):
        self.active_time_value = 0.25
        self.read_count = 0
        self.fail = False

    def set_active_time_value(self, value):
        """Select an available timestamp."""
        self.active_time_value = value

    def read(self):
        """Return the current mesh or simulate a read error."""
        self.read_count += 1
        if self.fail:
            raise RuntimeError('Synthetic read failure')
        return meshes[self.time_values.index(self.active_time_value)]

reader = SyntheticReader()
lazy = TimeSeries.from_reader(reader)
assert reader.read_count == 0
assert lazy[2].n_points == meshes[2].n_points
assert reader.read_count == 1
assert reader.active_time_value == 0.25
reader.fail = True
expect_error(RuntimeError, lambda: lazy[0])
assert reader.active_time_value == 0.25
print('Lazy loading and reader restoration passed')

## View all snapshots

In [ ]:
pl = pv.Plotter(shape=(1, 3), window_size=(1000, 350))
for index, time in enumerate(series.times):
    pl.subplot(0, index)
    pl.add_mesh(series[index], color='lightblue', show_edges=True)
    pl.add_text(f't = {time}')
    pl.camera_position = 'iso'
pl.show(jupyter_backend='static')